# Larkfield Parcels database generator

Larkfield Parcels is an invented regional courier with eight depots in the East of England and
the South Midlands. This notebook makes every row of `larkfield_parcels.db` using random numbers
(NumPy) and fake names and companies (Faker, UK locale). No real data is used, and the seed is fixed
so running it again gives the same file.

Order of work:

1. Settings
2. Price grid and depots
3. Vans and drivers
4. Business customers (senders)
5. Driver shifts
6. Consignments
7. Delivery attempts
8. The SQL schema
9. Deliberate data problems
10. Loading into SQLite
11. Checks and example queries

In [33]:
try:
    import faker
except ImportError:
    %pip install -q faker

## 1. Settings

Everything that controls the size and shape of the data is kept together here.

In [34]:
import os
import sqlite3
import numpy as np
import pandas as pd
from faker import Faker

SEED = 9274
rng = np.random.default_rng(SEED)
fake = Faker('en_GB')
Faker.seed(SEED)

DB_NAME = 'larkfield_parcels.db'
FIRST_DAY, LAST_DAY = '2026-03-02', '2026-09-30'      # seven months of trading
N_SENDERS = 700
N_CONSIGNMENTS = 16000

# depot code: (town, region, year opened, parcels sorted per hour, relative volume)
DEPOTS = {
    'LTN': ('Luton', 'East of England', 2009, 2400, 1.30),
    'STV': ('Stevenage', 'East of England', 2012, 1800, 1.00),
    'CBG': ('Cambridge', 'East of England', 2015, 2000, 1.15),
    'PBO': ('Peterborough', 'East of England', 2011, 2600, 1.25),
    'IPS': ('Ipswich', 'East of England', 2018, 1500, 0.80),
    'NWH': ('Norwich', 'East of England', 2016, 1600, 0.85),
    'MKY': ('Milton Keynes', 'South East', 2010, 3000, 1.45),
    'NTH': ('Northampton', 'East Midlands', 2019, 1700, 0.90),
}
POSTCODE_AREAS = {'LTN': ['LU1','LU2','LU3','LU4','LU5','LU6','LU7'], 'STV': ['SG1','SG2','SG4','SG5','SG6'],
                  'CBG': ['CB1','CB2','CB3','CB4','CB5','CB22','CB24'], 'PBO': ['PE1','PE2','PE3','PE4','PE7'],
                  'IPS': ['IP1','IP2','IP3','IP4','IP5'], 'NWH': ['NR1','NR2','NR3','NR4','NR5','NR6'],
                  'MKY': ['MK1','MK3','MK5','MK8','MK9','MK10','MK14'], 'NTH': ['NN1','NN2','NN3','NN4','NN5']}

SERVICES = ['Economy', 'Standard', 'Next day', 'Before 10am']                    # slowest to fastest
SERVICE_MIX = [0.22, 0.41, 0.30, 0.07]
DAYS_TO_FIRST_ATTEMPT = {'Economy': (3, 4), 'Standard': (2, 2), 'Next day': (1, 1), 'Before 10am': (1, 1)}
WEIGHT_BANDS = ['Up to 2 kg', '2 to 5 kg', '5 to 10 kg', '10 to 20 kg', '20 to 30 kg']   # ordered
BAND_EDGES = [0, 2, 5, 10, 20, 30]
TIERS = ['Pay as you go', 'Bronze', 'Silver', 'Gold']                            # account level, ordered
TIER_DISCOUNT = {'Pay as you go': 0.0, 'Bronze': 0.06, 'Silver': 0.12, 'Gold': 0.18}
VAN_CLASSES = {'Small van': (650, 0.35), 'Medium van': (1100, 0.40), 'Large van': (1400, 0.20), '7.5t lorry': (2800, 0.05)}
MONTHLY_MEAN_C = {3: 7.5, 4: 10.0, 5: 13.2, 6: 16.3, 7: 18.6, 8: 18.3, 9: 15.4}

print("RNG:", rng)
print("NumPy:", np.__version__)
print("work_days:", 'work_days' in globals())
print("N_CONSIGNMENTS:", 'N_CONSIGNMENTS' in globals())
print("senders:", 'senders' in globals())

RNG: Generator(PCG64)
NumPy: 2.4.6
work_days: True
N_CONSIGNMENTS: True
senders: True


## 2. Price grid and depots

The price grid is one row per service and weight band, so it has a two-column primary key.
Each consignment points back to its grid row with a composite foreign key. Prices go up with
both speed and weight.

In [35]:
base_by_band = np.array([3.60, 5.10, 7.40, 10.90, 15.80])        # Economy price per band
speed_uplift = np.array([1.00, 1.22, 1.55, 2.35])                 # multiplier per service
grid = [(s, b, round(float(base_by_band[j] * speed_uplift[i]), 2))
        for i, s in enumerate(SERVICES) for j, b in enumerate(WEIGHT_BANDS)]
service_rates = pd.DataFrame(grid, columns=['service_level', 'weight_band', 'price_gbp'])

depot_codes = list(DEPOTS)
depots = pd.DataFrame({
    'depot_id': np.arange(1, len(depot_codes) + 1),
    'depot_code': depot_codes,
    'town': [DEPOTS[c][0] for c in depot_codes],
    'region': [DEPOTS[c][1] for c in depot_codes],
    'opened_year': [DEPOTS[c][2] for c in depot_codes],
    'sort_capacity_per_hour': [DEPOTS[c][3] for c in depot_codes],
})
volume = np.array([DEPOTS[c][4] for c in depot_codes])
volume_p = volume / volume.sum()
depots

,depot_id,depot_code,town,region,opened_year,sort_capacity_per_hour
0,1,LTN,Luton,East of England,2009,2400
1,2,STV,Stevenage,East of England,2012,1800
2,3,CBG,Cambridge,East of England,2015,2000
3,4,PBO,Peterborough,East of England,2011,2600
4,5,IPS,Ipswich,East of England,2018,1500
5,6,NWH,Norwich,East of England,2016,1600
6,7,MKY,Milton Keynes,South East,2010,3000
7,8,NTH,Northampton,East Midlands,2019,1700


## 3. Vans and drivers

Each depot gets roughly ten vehicles per unit of volume. Number plates follow the real UK format,
where the two digits show when the vehicle was first registered (for example 24 for March to
August 2024 and 74 for September 2024 to February 2025). Electric vans are newer on average.

In [36]:
fleet_size = np.round(volume * 9 + rng.normal(0, 1, len(volume))).astype(int)
v_depot = np.repeat(depots.depot_id.values, fleet_size)
nv = len(v_depot)
v_class = rng.choice(list(VAN_CLASSES), size=nv, p=[v[1] for v in VAN_CLASSES.values()])
v_fuel = rng.choice(['Diesel', 'Electric', 'Hybrid'], size=nv, p=[0.58, 0.30, 0.12])
v_fuel = np.where(v_class == '7.5t lorry', 'Diesel', v_fuel)            # no electric lorries in this fleet

# electric vans are younger: age in years drawn from different gammas
age_years = np.where(v_fuel == 'Electric', rng.gamma(2.0, 0.9, nv), rng.gamma(2.6, 1.5, nv))
first_reg = pd.Timestamp(FIRST_DAY) - pd.to_timedelta((age_years * 365).astype(int), unit='D')

def plate_for(reg_date, letters):
    # UK age identifier: March-August uses the year, September-February uses year + 50
    yy = reg_date.year % 100
    if reg_date.month >= 9:
        ident = yy + 50
    elif reg_date.month <= 2:
        ident = (yy - 1) % 100 + 50
    else:
        ident = yy
    return f'{letters[:2]}{ident:02d} {letters[2:]}'

alphabet = list('ABCDEFGHJKLMNOPRSTUVWXY')
letter_sets = [''.join(rng.choice(alphabet, 5)) for _ in range(nv)]
vehicles = pd.DataFrame({
    'vehicle_id': np.arange(1, nv + 1),
    'depot_id': v_depot,
    'fleet_no': np.concatenate([np.arange(1, k + 1) for k in fleet_size]),
    'registration': [plate_for(d, l) for d, l in zip(first_reg, letter_sets)],
    'vehicle_class': v_class,
    'fuel_type': v_fuel,
    'payload_kg': [int(VAN_CLASSES[c][0] + rng.normal(0, 40)) for c in v_class],
    'first_registered': first_reg.strftime('%Y-%m-%d'),
})
assert vehicles.registration.is_unique

# drivers: a few more drivers than vehicles at each depot to cover days off
d_count = np.round(fleet_size * 1.2).astype(int)
d_depot = np.repeat(depots.depot_id.values, d_count)
nd = len(d_depot)
service_yrs = rng.gamma(2.2, 1.8, nd)
drivers = pd.DataFrame({
    'driver_id': np.arange(1, nd + 1),
    'depot_id': d_depot,
    'first_name': [fake.first_name() for _ in range(nd)],
    'last_name': [fake.last_name() for _ in range(nd)],
    'contract_type': rng.choice(['Employed', 'Self-employed', 'Agency'], size=nd, p=[0.55, 0.35, 0.10]),
    'started_on': (pd.Timestamp(FIRST_DAY) - pd.to_timedelta((service_yrs * 365).astype(int), unit='D')).strftime('%Y-%m-%d'),
})
print(nv, 'vehicles,', nd, 'drivers')

79 vehicles, 94 drivers


## 4. Business customers (senders)

Senders are small UK businesses from Faker. Repeated company names are dropped first, so the only
duplicate accounts are the deliberate ones in step 9. Bigger accounts (Silver and Gold) send more
parcels and get a bigger discount.

In [37]:
names = pd.Series([fake.company() for _ in range(int(N_SENDERS * 1.3))])
names = names[~names.str.lower().duplicated()].head(N_SENDERS).reset_index(drop=True)
ns = len(names)
home = rng.choice(depots.depot_id.values, size=ns, p=volume_p)
tier = rng.choice(TIERS, size=ns, p=[0.38, 0.30, 0.21, 0.11])

def contact_email(company):
    stem = ''.join(ch for ch in company.lower() if ch.isalpha())[:18]
    return f"{rng.choice(['orders', 'dispatch', 'hello', 'info', 'accounts'])}@{stem}.co.uk"

senders = pd.DataFrame({
    'sender_id': np.arange(1, ns + 1),
    'business_name': names.values,
    'contact_email': [contact_email(c) for c in names],
    'phone': ['01' + str(rng.integers(100, 999)) + ' ' + str(rng.integers(100000, 999999)) for _ in range(ns)],
    'postcode_district': [rng.choice(POSTCODE_AREAS[depot_codes[h - 1]]) for h in home],
    'account_tier': tier,
    'home_depot_id': home,
    'opened_on': (pd.Timestamp(FIRST_DAY) - pd.to_timedelta(rng.integers(20, 2900, ns), unit='D')).strftime('%Y-%m-%d'),
})
# how many parcels each sender books, relative to others
send_weight = np.select([tier == 'Gold', tier == 'Silver', tier == 'Bronze'], [9.0, 4.0, 2.0], default=1.0) \
              * rng.gamma(1.5, 1.0, ns)
senders.account_tier.value_counts()

,count
account_tier,
Pay as you go,277
Bronze,192
Silver,138
Gold,93


## 5. Driver shifts

Drivers work Monday to Saturday. On each day a driver is on shift with probability 0.8, and the
depot's vehicles are shared out among that day's drivers so that no van is used twice in a day.
Odometer readings carry on from one shift to the next for each van, and electric vans cover
fewer miles when it is cold.

In [38]:
work_days = pd.bdate_range(FIRST_DAY, LAST_DAY, freq='C', weekmask='Mon Tue Wed Thu Fri Sat')
shift_rows = []
for day in work_days:
    for dep in depots.depot_id.values:
        on = drivers.driver_id.values[(drivers.depot_id.values == dep) & (rng.random(nd) < 0.8)]
        vans = rng.permutation(vehicles.vehicle_id.values[vehicles.depot_id.values == dep])
        k = min(len(on), len(vans))                      # a driver without a free van stays in the depot
        for drv, van in zip(on[:k], vans[:k]):
            shift_rows.append((drv, day, van))
shifts = pd.DataFrame(shift_rows, columns=['driver_id', 'shift_date', 'vehicle_id'])

month_mean = shifts.shift_date.dt.month.map(MONTHLY_MEAN_C).values
shifts['ambient_temp_c'] = np.round(month_mean + rng.normal(0, 3.4, len(shifts)), 1)
shifts['start_time'] = (shifts.shift_date + pd.to_timedelta(rng.normal(7.1, 0.35, len(shifts)) * 3600, unit='s')
                        ).dt.strftime('%H:%M')

fuel = shifts.vehicle_id.map(vehicles.set_index('vehicle_id').fuel_type)
km = rng.gamma(9.0, 13.5, len(shifts))                    # about 120 km a day on average
km = np.where(fuel == 'Electric', km * np.clip(0.75 + shifts.ambient_temp_c.values / 60, 0.7, 1.0), km)
shifts['distance_km'] = np.round(km).astype(int)

shifts = shifts.sort_values(['vehicle_id', 'shift_date']).reset_index(drop=True)
start_odo = shifts.vehicle_id.map(pd.Series((age_years * 21000 + rng.normal(0, 3000, nv)).clip(500), index=vehicles.vehicle_id))
before = shifts.groupby('vehicle_id').distance_km.cumsum() - shifts.distance_km
shifts['start_odometer_km'] = (start_odo + before).round().astype(int)
shifts['end_odometer_km'] = shifts.start_odometer_km + shifts.distance_km
shifts['shift_date'] = shifts.shift_date.dt.strftime('%Y-%m-%d')
print(len(shifts), 'shifts')


13349 shifts


## 6. Consignments

Booking times are spread over the working week with a peak in the late afternoon, when online
shops print their labels. Parcel weight follows a log-normal shape, so most parcels are light and
a few are heavy. Box length grows with weight. The price comes from the grid for that service and
weight band, less the sender's account discount.

In [39]:

book_day = rng.choice(work_days[:-4], size=N_CONSIGNMENTS)                 # last few days left for delivery
book_hour = np.clip(rng.normal(15.2, 2.6, N_CONSIGNMENTS), 7, 21.5)
booked_at = pd.DatetimeIndex(book_day) + pd.to_timedelta((book_hour * 3600).astype(int), unit='s')

sender = rng.choice(senders.sender_id.values, size=N_CONSIGNMENTS, p=send_weight / send_weight.sum())
sender_tier = senders.account_tier.values[sender - 1]
origin = senders.home_depot_id.values[sender - 1]
service = rng.choice(SERVICES, size=N_CONSIGNMENTS, p=SERVICE_MIX)

weight = np.round(np.clip(rng.lognormal(0.55, 0.95, N_CONSIGNMENTS), 0.05, 29.9), 2)
band = pd.cut(weight, BAND_EDGES, labels=WEIGHT_BANDS, right=True).astype(str)
length = np.round(np.clip(18 + 11 * np.sqrt(weight) + rng.normal(0, 6, N_CONSIGNMENTS), 10, 160)).astype(int)
value = np.round(np.exp(rng.normal(3.4, 0.9, N_CONSIGNMENTS)) + weight * 4, 2)
dest = [rng.choice(POSTCODE_AREAS[depot_codes[o - 1]]) for o in origin]

price = pd.DataFrame({'service_level': service, 'weight_band': band}).merge(
    service_rates, on=['service_level', 'weight_band'], how='left').price_gbp.values
discount = np.array([TIER_DISCOUNT[t] for t in sender_tier])

order = np.argsort(booked_at.values)
consignments = pd.DataFrame({
    'sender_id': sender[order], 'origin_depot_id': origin[order],
    'service_level': service[order], 'weight_band': band[order],
    'booked_at': booked_at[order].strftime('%Y-%m-%d %H:%M:%S'),
    'weight_kg': weight[order], 'length_cm': length[order],
    'declared_value_gbp': value[order],
    'dest_postcode_district': np.array(dest)[order],
    'recipient_phone': ['07' + str(rng.integers(100, 999)) + ' ' + str(rng.integers(100000, 999999)) for _ in order],
    'price_gbp': np.round(price[order] * (1 - discount[order]), 2),
})


consignments.insert(0, 'consignment_id', np.arange(1, N_CONSIGNMENTS + 1))

# Generate unique tracking numbers without creating a huge array
tracking_ids = rng.integers(10**9, 10**10, size=N_CONSIGNMENTS)

while len(np.unique(tracking_ids)) < N_CONSIGNMENTS:
    unique_ids = np.unique(tracking_ids)
    missing = N_CONSIGNMENTS - len(unique_ids)
    extra_ids = rng.integers(10**9, 10**10, size=missing)
    tracking_ids = np.concatenate((unique_ids, extra_ids))

tracking_ids = np.unique(tracking_ids)

consignments.insert(1, 'tracking_no', ['LK' + str(x) for x in tracking_ids])


print("Consignments generated successfully!")
print("Total consignments:", len(consignments))
print(consignments.head())
print(consignments.service_level.value_counts().to_dict())

Consignments generated successfully!
Total consignments: 16000
   consignment_id   tracking_no  sender_id  origin_depot_id service_level  \
0               1  LK1001099326        672                3   Before 10am   
1               2  LK1001168511        413                6      Standard   
2               3  LK1001678651        277                8   Before 10am   
3               4  LK1002130836        498                7      Standard   
4               5  LK1003063818        379                7      Standard   

  weight_band            booked_at  weight_kg  length_cm  declared_value_gbp  \
0  5 to 10 kg  2026-03-02 08:18:32       6.29         44              111.50   
1  Up to 2 kg  2026-03-02 10:45:53       0.42         23               78.68   
2   2 to 5 kg  2026-03-02 11:00:30       2.61         34               20.70   
3   2 to 5 kg  2026-03-02 11:11:19       3.62         44               29.00   
4  Up to 2 kg  2026-03-02 11:40:28       1.20         30               18.

## 7. Delivery attempts

The first attempt is made one to four working days after booking, depending on the service.
About 87% of first attempts succeed. A failed parcel is tried again the next working day, up to
three times, and then goes back to the sender. Each attempt is given to a driver who is actually
on shift at that depot that day, and it uses that driver's van.

In [40]:
shift_index = shifts[['driver_id', 'vehicle_id', 'shift_date']].copy()
shift_index['depot_id'] = shift_index.driver_id.map(drivers.set_index('driver_id').depot_id)
shift_index['slot'] = shift_index.groupby(['depot_id', 'shift_date']).cumcount()
slots = shift_index.groupby(['depot_id', 'shift_date']).size()
day_pos = {d.strftime('%Y-%m-%d'): i for i, d in enumerate(work_days)}
day_list = [d.strftime('%Y-%m-%d') for d in work_days]

lag = np.array([rng.integers(*np.array(DAYS_TO_FIRST_ATTEMPT[s]) + [0, 1]) for s in consignments.service_level])
start_pos = np.array([day_pos.get(b[:10], 0) for b in consignments.booked_at]) + lag
success_p = [0.87, 0.81, 0.76]

attempt_rows = []
remaining = consignments[['consignment_id', 'origin_depot_id', 'service_level']].copy()
remaining['pos'] = np.minimum(start_pos, len(day_list) - 1)
for attempt in (1, 2, 3):
    if remaining.empty:
        break
    a = remaining.copy()
    a['attempt_no'] = attempt
    a['shift_date'] = [day_list[p] for p in a.pos]
    key = list(zip(a.origin_depot_id, a.shift_date))
    a['slot'] = (rng.random(len(a)) * np.array([slots.get(k, 1) for k in key])).astype(int)
    a = a.merge(shift_index, left_on=['origin_depot_id', 'shift_date', 'slot'],
                right_on=['depot_id', 'shift_date', 'slot'], how='left')
    hours = np.where(a.service_level == 'Before 10am', rng.uniform(7.6, 9.95, len(a)), rng.uniform(8.2, 19.0, len(a)))
    a['attempted_at'] = (pd.to_datetime(a.shift_date) + pd.to_timedelta((hours * 3600).astype(int), unit='s')
                         ).dt.strftime('%Y-%m-%d %H:%M:%S')
    ok = rng.random(len(a)) < success_p[attempt - 1]
    a['outcome'] = np.where(ok, rng.choice(['Delivered to recipient', 'Left with neighbour', 'Left in safe place'],
                                           size=len(a), p=[0.68, 0.19, 0.13]),
                            rng.choice(['Card left', 'Refused'], size=len(a), p=[0.93, 0.07]))
    a['stop_distance_km'] = np.round(rng.gamma(2.0, 1.6, len(a)), 1)
    attempt_rows.append(a)
    retry = ~ok & (a.outcome.values != 'Refused')
    remaining = a.loc[retry, ['consignment_id', 'origin_depot_id', 'service_level']].copy()
    remaining['pos'] = np.minimum(a.loc[retry, 'shift_date'].map(day_pos).values + 1, len(day_list) - 1)

attempts = pd.concat(attempt_rows, ignore_index=True)
attempts = attempts[['consignment_id', 'attempt_no', 'driver_id', 'vehicle_id', 'attempted_at', 'outcome', 'stop_distance_km']]

last = attempts.sort_values('attempt_no').groupby('consignment_id').tail(1).set_index('consignment_id')
delivered = last.outcome.isin(['Delivered to recipient', 'Left with neighbour', 'Left in safe place'])
consignments['status'] = np.where(consignments.consignment_id.map(delivered), 'Delivered', 'Returned to sender')
lost = rng.random(len(consignments)) < 0.002                       # a handful go missing
consignments.loc[lost, 'status'] = 'Lost'
attempts = attempts[~attempts.consignment_id.isin(consignments.consignment_id[lost])]   # lost before reaching a van
print(len(attempts), 'attempts;', consignments.status.value_counts().to_dict())

18250 attempts; {'Delivered': 15703, 'Returned to sender': 265, 'Lost': 32}


## 8. The SQL schema

The DDL sits here, just before loading, so it can be read alongside the data it has to hold.
There are eight tables with eleven foreign keys. One of those keys is composite: consignments
point to service_rates using two columns. There are three composite primary keys (service_rates,
delivery_attempts and driver_shifts) and one composite UNIQUE key (depot and fleet number).

In [41]:
SCHEMA_SQL = """
PRAGMA foreign_keys = ON;

CREATE TABLE depots (
    depot_id                INTEGER PRIMARY KEY,
    depot_code              TEXT    NOT NULL UNIQUE CHECK (depot_code GLOB '[A-Z][A-Z][A-Z]'),
    town                    TEXT    NOT NULL,
    region                  TEXT    NOT NULL CHECK (region IN ('East of England','South East','East Midlands')),
    opened_year             INTEGER NOT NULL CHECK (opened_year > 1980 AND opened_year <= 2026),
    sort_capacity_per_hour  INTEGER NOT NULL CHECK (sort_capacity_per_hour BETWEEN 100 AND 20000)
);

CREATE TABLE service_rates (
    service_level  TEXT NOT NULL CHECK (service_level IN ('Economy','Standard','Next day','Before 10am')),
    weight_band    TEXT NOT NULL CHECK (weight_band IN ('Up to 2 kg','2 to 5 kg','5 to 10 kg','10 to 20 kg','20 to 30 kg')),
    price_gbp      REAL NOT NULL CHECK (price_gbp > 0),
    PRIMARY KEY (service_level, weight_band)
);

CREATE TABLE vehicles (
    vehicle_id        INTEGER PRIMARY KEY,
    depot_id          INTEGER NOT NULL REFERENCES depots(depot_id) ON DELETE RESTRICT,
    fleet_no          INTEGER NOT NULL CHECK (fleet_no BETWEEN 1 AND 99),
    registration      TEXT    NOT NULL UNIQUE CHECK (length(registration) = 8 AND registration GLOB '[A-Z][A-Z][0-9][0-9] [A-Z]*'),
    vehicle_class     TEXT    NOT NULL CHECK (vehicle_class IN ('Small van','Medium van','Large van','7.5t lorry')),
    fuel_type         TEXT    NOT NULL CHECK (fuel_type IN ('Diesel','Electric','Hybrid')),
    payload_kg        INTEGER NOT NULL CHECK (payload_kg BETWEEN 300 AND 4000),
    first_registered  TEXT    NOT NULL CHECK (strftime('%Y-%m-%d', first_registered) = first_registered),
    UNIQUE (depot_id, fleet_no)
);

CREATE TABLE drivers (
    driver_id      INTEGER PRIMARY KEY,
    depot_id       INTEGER NOT NULL REFERENCES depots(depot_id) ON DELETE RESTRICT,
    first_name     TEXT    NOT NULL CHECK (length(first_name) BETWEEN 2 AND 24 AND first_name NOT GLOB '*[0-9]*'),
    last_name      TEXT    NOT NULL CHECK (length(last_name) BETWEEN 2 AND 32 AND last_name NOT GLOB '*[0-9]*'),
    contract_type  TEXT    NOT NULL CHECK (contract_type IN ('Employed','Self-employed','Agency')),
    started_on     TEXT    NOT NULL CHECK (started_on GLOB '[12][0-9][0-9][0-9]-[01][0-9]-[0-3][0-9]')
);

CREATE TABLE senders (
    sender_id          INTEGER PRIMARY KEY,
    business_name      TEXT    NOT NULL CHECK (length(trim(business_name)) > 0),
    contact_email      TEXT    NOT NULL CHECK (contact_email LIKE '%_@_%._%'),
    phone              TEXT    CHECK (phone IS NULL OR phone GLOB '01[0-9][0-9][0-9] [0-9][0-9][0-9][0-9][0-9][0-9]'),
    postcode_district  TEXT    NOT NULL CHECK (length(postcode_district) IN (3, 4) AND postcode_district GLOB '[A-Z][A-Z][0-9]*' AND postcode_district NOT GLOB '*[^A-Z0-9]*'),
    account_tier       TEXT    NOT NULL DEFAULT 'Pay as you go'
                               CHECK (account_tier IN ('Pay as you go','Bronze','Silver','Gold')),
    home_depot_id      INTEGER NOT NULL REFERENCES depots(depot_id) ON DELETE RESTRICT,
    opened_on          TEXT    NOT NULL CHECK (date(opened_on) = opened_on)
);

CREATE TABLE consignments (
    consignment_id          INTEGER PRIMARY KEY,
    tracking_no             TEXT    NOT NULL UNIQUE CHECK (tracking_no GLOB 'LK[0-9]*' AND length(tracking_no) = 12),
    sender_id               INTEGER NOT NULL REFERENCES senders(sender_id) ON DELETE RESTRICT,
    origin_depot_id         INTEGER NOT NULL REFERENCES depots(depot_id) ON DELETE RESTRICT,
    service_level           TEXT    NOT NULL,
    weight_band             TEXT    NOT NULL,
    booked_at               TEXT    NOT NULL CHECK (strftime('%Y-%m-%d %H:%M:%S', booked_at) = booked_at),
    weight_kg               REAL    NOT NULL CHECK (weight_kg > 0 AND weight_kg <= 30),
    length_cm               INTEGER NOT NULL CHECK (length_cm BETWEEN 5 AND 200),
    declared_value_gbp      REAL    CHECK (declared_value_gbp IS NULL OR declared_value_gbp BETWEEN 0 AND 5000),
    dest_postcode_district  TEXT    NOT NULL CHECK (dest_postcode_district GLOB '[A-Z][A-Z][0-9]*'),
    recipient_phone         TEXT    CHECK (recipient_phone IS NULL OR recipient_phone GLOB '07[0-9][0-9][0-9] [0-9]*'),
    price_gbp               REAL    NOT NULL CHECK (price_gbp > 0),
    status                  TEXT    NOT NULL DEFAULT 'Delivered'
                                    CHECK (status IN ('Delivered','Returned to sender','Lost')),
    FOREIGN KEY (service_level, weight_band) REFERENCES service_rates(service_level, weight_band) ON UPDATE CASCADE
);

CREATE TABLE driver_shifts (
    driver_id          INTEGER NOT NULL REFERENCES drivers(driver_id) ON DELETE CASCADE,
    shift_date         TEXT    NOT NULL CHECK (date(shift_date) = shift_date),
    vehicle_id         INTEGER NOT NULL REFERENCES vehicles(vehicle_id) ON UPDATE CASCADE ON DELETE RESTRICT,
    start_time         TEXT    NOT NULL CHECK (length(start_time) = 5 AND substr(start_time, 3, 1) = ':' AND CAST(substr(start_time, 1, 2) AS INTEGER) < 24),
    ambient_temp_c     REAL    NOT NULL CHECK (ambient_temp_c BETWEEN -20 AND 40),
    distance_km        INTEGER NOT NULL CHECK (distance_km BETWEEN 0 AND 800),
    start_odometer_km  INTEGER NOT NULL CHECK (start_odometer_km >= 0),
    end_odometer_km    INTEGER CHECK (end_odometer_km IS NULL OR end_odometer_km >= start_odometer_km),
    PRIMARY KEY (driver_id, shift_date)
);

CREATE TABLE delivery_attempts (
    consignment_id    INTEGER NOT NULL REFERENCES consignments(consignment_id) ON DELETE CASCADE,
    attempt_no        INTEGER NOT NULL CHECK (attempt_no BETWEEN 1 AND 3),
    driver_id         INTEGER NOT NULL REFERENCES drivers(driver_id) ON DELETE RESTRICT,
    vehicle_id        INTEGER NOT NULL CONSTRAINT fk_attempt_van REFERENCES vehicles(vehicle_id) ON DELETE RESTRICT,
    attempted_at      TEXT    NOT NULL CHECK (datetime(attempted_at) = attempted_at),
    outcome           TEXT    NOT NULL CHECK (outcome IN ('Delivered to recipient','Left with neighbour',
                                                         'Left in safe place','Card left','Refused')),
    stop_distance_km  REAL    NOT NULL CHECK (stop_distance_km >= 0),
    PRIMARY KEY (consignment_id, attempt_no)
);

CREATE INDEX idx_cons_sender ON consignments(sender_id);
CREATE INDEX idx_att_driver  ON delivery_attempts(driver_id, attempted_at);
"""

## 9. Deliberate data problems

A courier's records are never perfect, so these problems are added **on purpose**, with fixed
rates, before loading. The cell prints how many rows each one touched.

| Problem | Where | Rate | What causes it in real life |
|---|---|---|---|
| Missing | senders.phone | 4% | account opened online without a phone number |
| Missing | consignments.declared_value_gbp | 5% | value is optional when booking |
| Missing | consignments.recipient_phone | 3.5% | shop did not pass the buyer's number on |
| Missing | driver_shifts.end_odometer_km | 1.5% | driver forgot to log the reading |
| Duplicate | senders | 2% | the same business opened a second account |
| Duplicate | consignments | 0.6% | label printed twice for one parcel |

In [42]:
problem_log = {}

def blank_out(df, col, rate):
    # pick rows at random and empty one column
    rows = rng.choice(df.index.values, size=int(round(len(df) * rate)), replace=False)
    df.loc[rows, col] = None
    problem_log[f'missing {col}'] = len(rows)

blank_out(senders, 'phone', 0.04)
blank_out(consignments, 'declared_value_gbp', 0.05)
blank_out(consignments, 'recipient_phone', 0.035)
blank_out(shifts, 'end_odometer_km', 0.015)

# second accounts for the same business: Ltd and Limited swapped, or the case changed
twins = senders.sample(frac=0.02, random_state=SEED).copy()
style = np.arange(len(twins)) % 3
twins['business_name'] = np.where(style == 0, twins.business_name.str.upper(),
                          np.where(style == 1, twins.business_name.str.replace('Ltd', 'Limited', regex=False) + ' ',
                                   twins.business_name.str.lower()))
twins['account_tier'] = 'Pay as you go'
twins['opened_on'] = (pd.to_datetime(twins.opened_on) + pd.to_timedelta(rng.integers(30, 500, len(twins)), unit='D')
                      ).clip(upper=pd.Timestamp(FIRST_DAY)).dt.strftime('%Y-%m-%d')
problem_log['duplicate senders'] = len(twins)
problem_log['  original sender ids'] = sorted(twins.sender_id.tolist())
twins['sender_id'] = np.arange(senders.sender_id.max() + 1, senders.sender_id.max() + 1 + len(twins))
senders = pd.concat([senders, twins], ignore_index=True)

# a label printed twice: same parcel details, new id and tracking number, never attempted
reprint = consignments.sample(frac=0.006, random_state=SEED).copy()
problem_log['duplicate consignments'] = len(reprint)
problem_log['  first original consignment ids'] = sorted(reprint.consignment_id.tolist())[:8]
reprint['consignment_id'] = np.arange(consignments.consignment_id.max() + 1,
                                      consignments.consignment_id.max() + 1 + len(reprint))

existing_tracking = set(consignments['tracking_no'].dropna())

new_tracking = []

while len(new_tracking) < len(reprint):
    candidates = rng.integers(10**9, 10**10, size=len(reprint) * 2)

    for x in candidates:
        tracking = 'LK' + str(x)

        if tracking not in existing_tracking:
            new_tracking.append(tracking)
            existing_tracking.add(tracking)

        if len(new_tracking) == len(reprint):
            break

reprint['tracking_no'] = new_tracking
consignments = pd.concat([consignments, reprint], ignore_index=True)

for k, v in problem_log.items():
    print(f'{k:36s} {v}')

missing phone                        28
missing declared_value_gbp           800
missing recipient_phone              560
missing end_odometer_km              200
duplicate senders                    14
  original sender ids                [93, 141, 174, 193, 208, 212, 315, 383, 405, 421, 490, 585, 608, 700]
duplicate consignments               96
  first original consignment ids     [202, 574, 706, 731, 758, 959, 1195, 1222]


## 10. Loading into SQLite

In [43]:
if os.path.exists(DB_NAME):
    os.remove(DB_NAME)
con = sqlite3.connect(DB_NAME)
con.executescript(SCHEMA_SQL)
con.execute('PRAGMA foreign_keys = ON')

def push(df, table):
    # NaN becomes NULL, then insert all rows in one call
    rows = df.astype(object).where(df.notna(), None).itertuples(index=False, name=None)
    holes = ','.join('?' * df.shape[1])
    con.executemany(f'INSERT INTO {table} ({",".join(df.columns)}) VALUES ({holes})', list(rows))

push(depots, 'depots')
push(service_rates, 'service_rates')
push(vehicles, 'vehicles')
push(drivers, 'drivers')
push(senders, 'senders')
push(consignments, 'consignments')
push(shifts[['driver_id', 'shift_date', 'vehicle_id', 'start_time', 'ambient_temp_c', 'distance_km',
             'start_odometer_km', 'end_odometer_km']], 'driver_shifts')
push(attempts, 'delivery_attempts')
con.commit()

tables = ['depots', 'service_rates', 'vehicles', 'drivers', 'senders', 'consignments', 'driver_shifts', 'delivery_attempts']
pd.Series({t: con.execute(f'SELECT COUNT(*) FROM {t}').fetchone()[0] for t in tables}, name='rows')

,rows
depots,8
service_rates,20
vehicles,79
drivers,94
senders,714
consignments,16096
driver_shifts,13349
delivery_attempts,18250


## 11. Checks and example queries

In [44]:
run = lambda q: pd.read_sql_query(q, con)
print('integrity:', con.execute('PRAGMA integrity_check').fetchone()[0])
print('foreign key problems:', con.execute('PRAGMA foreign_key_check').fetchall())
run('''
SELECT 'senders.phone' AS col, SUM(phone IS NULL) AS missing FROM senders
UNION ALL SELECT 'consignments.declared_value_gbp', SUM(declared_value_gbp IS NULL) FROM consignments
UNION ALL SELECT 'consignments.recipient_phone', SUM(recipient_phone IS NULL) FROM consignments
UNION ALL SELECT 'driver_shifts.end_odometer_km', SUM(end_odometer_km IS NULL) FROM driver_shifts
''')

integrity: ok
foreign key problems: []


,col,missing
0,senders.phone,29
1,consignments.declared_value_gbp,806
2,consignments.recipient_phone,565
3,driver_shifts.end_odometer_km,200


In [45]:
# the deliberate duplicates can be found again with plain SQL
run('''
SELECT
 (SELECT COUNT(*) - COUNT(DISTINCT lower(trim(replace(replace(business_name, 'Limited', 'Ltd'), 'LIMITED', 'LTD')))
                                   || postcode_district) FROM senders) AS duplicate_senders,
 (SELECT COUNT(*) FROM (SELECT 1 FROM consignments
     GROUP BY sender_id, booked_at, weight_kg, dest_postcode_district HAVING COUNT(*) > 1)) AS double_labels
''')

,duplicate_senders,double_labels
0,14,96


In [46]:
run('''
SELECT c.service_level, COUNT(*) AS parcels,
       ROUND(100.0 * SUM(a.outcome IN ('Delivered to recipient','Left with neighbour','Left in safe place'))
             / COUNT(*), 1) AS first_time_pct
FROM consignments c JOIN delivery_attempts a ON a.consignment_id = c.consignment_id AND a.attempt_no = 1
GROUP BY c.service_level
ORDER BY CASE c.service_level WHEN 'Economy' THEN 1 WHEN 'Standard' THEN 2 WHEN 'Next day' THEN 3 ELSE 4 END
''')

,service_level,parcels,first_time_pct
0,Economy,3485,86.5
1,Standard,6537,87.5
2,Next day,4831,86.7
3,Before 10am,1115,87.3


In [47]:
run('''
SELECT v.fuel_type, CASE WHEN s.ambient_temp_c < 8 THEN 'cold (<8C)' ELSE 'mild (8C+)' END AS weather,
       ROUND(AVG(s.distance_km), 1) AS avg_km_per_shift, COUNT(*) AS shifts
FROM driver_shifts s JOIN vehicles v USING (vehicle_id)
GROUP BY v.fuel_type, weather ORDER BY v.fuel_type, weather
''')

,fuel_type,weather,avg_km_per_shift,shifts
0,Diesel,cold (<8C),123.4,1074
1,Diesel,mild (8C+),122.8,6710
2,Electric,cold (<8C),102.2,620
3,Electric,mild (8C+),117.9,4119
4,Hybrid,cold (<8C),121.1,115
5,Hybrid,mild (8C+),123.7,711


In [48]:
con.close()
print(DB_NAME, round(os.path.getsize(DB_NAME) / 1e6, 2), 'MB')

larkfield_parcels.db 5.52 MB
